In [5]:

import requests
from bs4 import BeautifulSoup
import re
import pandas as pd

url = "https://wahapedia.ru/aos4/the-rules/quick-start-guide/"
response = requests.get(url)
soup = BeautifulSoup(response.content, "html.parser")



# Example: Get factions links on the page
links = [a['href'] for a in soup.find_all('a', href=True)]
faction_links = [link for link in links if "factions" in link]   # Gets all the faction links
faction_names = [link.rsplit('/', 1)[-1] for link in faction_links] # Gets all the faction names
print(faction_names)


parsed_faction_names = []
for x in faction_names:
    parsed_faction_names.append(x.replace("-", " ").title())



['cities-of-sigmar', 'daughters-of-khaine', 'fyreslayers', 'idoneth-deepkin', 'kharadron-overlords', 'lumineth-realm-lords', 'seraphon', 'stormcast-eternals', 'sylvaneth', 'beasts-of-chaos', 'blades-of-khorne', 'disciples-of-tzeentch', 'hedonites-of-slaanesh', 'maggotkin-of-nurgle', 'skaven', 'slaves-to-darkness', 'flesh-eater-courts', 'nighthaunt', 'ossiarch-bonereapers', 'soulblight-gravelords', 'bonesplitterz', 'gloomspite-gitz', 'ironjawz', 'kruleboyz', 'ogor-mawtribes', 'sons-of-behemat', 'endless-spells', 'stormcast-eternals']


In [7]:
def get_warscrolls_for_an_army(faction_links: str, faction_names: str):
    """
    Fetch unique warscroll links for a given faction from Wahapedia.

    Parameters:
        faction_link (str): The relative link to the faction page (e.g. '/aos3/factions/cities-of-sigmar/')
        faction_names (str): A string to identify the faction in links (e.g. 'cities-of-sigmar/')

    Returns:
        list[str]: Unique warscroll URLs for the faction
    """
    url = "https://wahapedia.ru" + faction_links
    response = requests.get(url)
    response.raise_for_status()  # raise error if request fails

    soup = BeautifulSoup(response.content, "html.parser")
    # Collect all links on the page
    warscrolls = [a['href'] for a in soup.find_all('a', href=True)]

    # Keep only links containing the faction slug
    warscrolls = [link for link in warscrolls if faction_names in link]

    # Deduplicate while preserving order
    unique_warscrolls = list(dict.fromkeys(warscrolls[1:]))


    #Removing regiments of renown

    
    #lines = str(soup).splitlines()
    #Present = []
    # Print any line that contains "Regiment"
    #for line in lines:
    #    if "Regiment" in line:
    #        Present.append(line)
            
            
            
            
            
    #Regiment_of_Renown = (Present[0].split("Regiment")[1].strip())
    
    #to_remove = []

    #for x in range(len(unique_warscrolls)):
    #    Name = unique_warscrolls[x].split("/")[-1]
    #    # Check if the name is in the Regiment of Renown text
    #    #print("Regiment_of_Renown:", Regiment_of_Renown)
    #    if str(Name) in str(Regiment_of_Renown):
    #        to_remove.append(unique_warscrolls[x])
    #        print("to_remove:", to_remove)
            
            
            
    # Now remove them safely
    #for item in to_remove:
    #    unique_warscrolls.remove(item)
    
    #target = "/aos4/factions/"   + faction_names
    #print("Target to remove:", target)
    #unique_warscrolls = [w for w in unique_warscrolls if w != target]
    #print("Unique warscrolls before Legends removal:", unique_warscrolls)
    
    
    
    
    
    
    
    

    
    
    Name = []
    for x in unique_warscrolls:
        Name.append(x.split("/")[-1])   
        
    

        
        
        
    
    
    
    # THis removes the "Legends" section from the warscrolls
    # It finds the line containing "Legends" and extracts the text after it.
    # Then it checks each warscroll name against this text to find the ones that are present
    # in the "Legends" section.
    # Finally, it prints the ordered names of the warscrolls that are present in the
    
    to_remove = []
    
    for word in "Regiment", "Legends":
        lines = str(soup).splitlines()
        Present = []
        for line in lines:
            if str(word) in line:
                Present.append(line)
                
        Legends_Line = (Present[0])
        
        
        
        
        Sentence_Containing_Legends = Legends_Line.split(str(word))[1:] #Splits the line at "Legends" and takes the part after it however there is only a legend on every second so drop the odd ones

        #Sentence_Containing_Legends = Sentence_Containing_Legends[1::2] #Keeps every second element starting from index 1 (the second element)


        for x in range(len(Sentence_Containing_Legends)):
            #print(Sentence_Containing_Legends[x], "Sentence Containing Legends")
            
            matches = []
            for n in Name:
                m = re.search(re.escape(n), Sentence_Containing_Legends[x])
                
                if m:
                    
                    matches.append((m.start(), n))  # store position + name

            

            # Sort by position of first appearance

            matches.sort(key=lambda x: x[0])
            
            
            # Extract the ordered names
            ordered_matches = [n for _, n in matches]
            
            
            #for x in matches:
            
            if word  == "Legends":
                if ordered_matches:
                    
                    to_remove.append(f"/aos4/factions/{faction_names}/{ordered_matches[1]}")
                    
            
            
            
            elif ordered_matches:
                
                for z in ordered_matches:
                    
                    to_remove.append(f"/aos4/factions/{faction_names}/{z}")

        to_remove = list(dict.fromkeys(to_remove))   
       
        
        
        
        
        unique_warscrolls = [w for w in unique_warscrolls if w not in to_remove]        # Removes all unwanted warscrolls

        
        
        
        unique_warscrolls = [w for w in unique_warscrolls if not w.endswith("warscrolls.html")]

    return unique_warscrolls



unique_warscrolls = get_warscrolls_for_an_army(faction_links[1], faction_names[1])
print(unique_warscrolls)
print(len(unique_warscrolls))

['/aos4/factions/daughters-of-khaine/Bloodwrack-Medusa', '/aos4/factions/daughters-of-khaine/Hag-Queen', '/aos4/factions/daughters-of-khaine/High-Gladiatrix', '/aos4/factions/daughters-of-khaine/Krethusa-the-Croneseer', '/aos4/factions/daughters-of-khaine/Melusai-Ironscale', '/aos4/factions/daughters-of-khaine/Morathi-Khaine', '/aos4/factions/daughters-of-khaine/Slaughter-Queen', '/aos4/factions/daughters-of-khaine/Scourge-of-Ghyran-Krethusa-the-Croneseer', '/aos4/factions/daughters-of-khaine/The-Shadow-Queen', '/aos4/factions/daughters-of-khaine/Bloodwrack-Shrine', '/aos4/factions/daughters-of-khaine/Hag-Queen-on-Cauldron-of-Blood', '/aos4/factions/daughters-of-khaine/Slaughter-Queen-on-Cauldron-of-Blood', '/aos4/factions/daughters-of-khaine/Scourge-of-Ghyran-Bloodwrack-Shrine', '/aos4/factions/daughters-of-khaine/Doomfire-Warlocks', '/aos4/factions/daughters-of-khaine/Blood-Sisters', '/aos4/factions/daughters-of-khaine/Blood-Stalkers', '/aos4/factions/daughters-of-khaine/Khainite-Sha

In [28]:
def removing_unwanted_words(ability: str, parsed_faction_names: list) -> str:
    """
    Remove unwanted words, faction names, grammar, and apply replacements in ability text.

    Parameters:
        ability (str): The ability text to clean.
        parsed_faction_names (list): List of faction names to remove.

    Returns:
        str: Cleaned ability text.
    """
    Timing = ability

    # Words to remove
    Words_to_remove =  [
        "Effect", "Declare", "This unit", "This model", "This unit and the target",
        "this unit", "this model", "Battle", "Phase", "For", "The", "while", "Within",
        "Are", "To", "Be", "a", "unit", "that", "friendly", "and", "this", "turn", 
        "is", "of", "target", "rest", "Any", "Per", "Army", "If", "They", "Has", "Range", 
        "From", "Score", "unites", "unit", "your", "weapons", "roll", "rolls", "You", "Declared", "Ability",
        "Characteristics", "Characteristic", "in", "Attacks", "With", "Its", "scores", "inflict", "inflicts",
        "each", "time", "cannot", "pick", "used", "can", "immediately", "used", "been", "resolved", "automatically",
        "it", "pick", "effects", "allocate"
    ]
    Words_to_remove += parsed_faction_names

    # Grammar/punctuation to remove
    Grammar_to_remove = ['"', "‘", "’", "“", "”", "(", ")", ",", ".", ";", ":", "-", "—", "!", "?"]
    
    # Replacements dictionary
    replacements = {
        "Add ": "+",
        "Subtract ": "-",
        "Multiply ": "*",
        "Divide ": "/",
        "Damage ": "DMG",
        "Wounds ": "W",
        "Attacks ": "AT",
        "Attack ": "AT",
        "Save ": "SV",
        "equal" : "=",
        "number" : "x",
        "wholly": "ww",
        "WARD": "WD",
        "visible": "vis",
        "INFANTRY": "INF",
        "contesting an objective": "contest",
        "Cavalrty": "CAV",
        "more than": ">",
        "less than": "<",
        "replacement": "rep",
        "half": "1/2"
        
    }

    # Remove unwanted words (case-insensitive)
    pattern = r'\b(?:' + '|'.join(re.escape(word) for word in Words_to_remove) + r')\b'
    Timing = re.sub(pattern, " ", Timing, flags=re.IGNORECASE)

    # Remove grammar/punctuation
    pattern = r'(?:' + '|'.join(re.escape(p) for p in Grammar_to_remove) + r')'
    Timing = re.sub(pattern, " ", Timing)

    # Apply replacements
    for old, new in sorted(replacements.items(), key=lambda x: len(x[0]), reverse=True):
        Timing = re.sub(re.escape(old), new, Timing, flags=re.IGNORECASE)

    # Clean up extra spaces
    Timing = re.sub(r"\s+", " ", Timing).strip()

    return Timing

def is_integer(s):
    try:
        int(s)
        return True
    except ValueError:
        return False



In [10]:
def combine_before_indices(lst, indices):
    result = []
    start = 0
    indices = sorted(indices)  # ensure indices are in order
    for idx in indices:
        # join elements from start to idx (not including idx)
        if start < idx:
            combined = ' '.join(lst[start:idx])
            result.append(combined)
        start = idx  # move start to current index
    # add the remaining elements after the last index
    if start < len(lst):
        result.append(' '.join(lst[start:]))
    return result

In [31]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup



def get_abilities_from_each_warscroll(faction_links: str, faction_names: str, parsed_faction_names: str, Text_Cleaning = False, Passive_Cleaning=False) -> pd.DataFrame:
    """
    Scrape abilities from all warscrolls for a given faction.

    Parameters:
        faction_links (list): List of faction page links.
        faction_names (list): List of faction names.
        parsed_faction_names (list): List of parsed faction names to clean text.

    Returns:
        pd.DataFrame: DataFrame with columns ["Warscroll", "Timing", "Name", "Description"].
    """
    Unit_Abilities_df = pd.DataFrame(columns=["Warscroll", "Name", "Timing", "Flavour_Text", "Description"])

    # Get warscroll links for the army
    unique_warscrolls = get_warscrolls_for_an_army(faction_links, faction_names)

    for x_unique_warscrolls in unique_warscrolls:
        url = "https://wahapedia.ru" + x_unique_warscrolls
        Warscroll_Name = x_unique_warscrolls.split("/")[-1]
        #print(f"Processing warscroll: {Warscroll_Name}")

        response = requests.get(url)
        soup = BeautifulSoup(response.content, "html.parser")
        ws_body = soup.find("div", class_="wsBody")

        results = []
        if ws_body:
            # Look at all child <div> inside wsBody
            for div in ws_body.find_all("div"):
                if div.find("b"):  # only keep divs that contain <b>
                    text = div.decode_contents()
                    

                    text = text.replace("<br/>", " $ ").replace("<br>", " $ ")
                    results.append(text)
                    
                    
            results = [re.sub(r"<(?!/?i\b)[^>]*>", " ", r) for r in results]
            
            
            
        abilities = []
        Acceptable_Starts = ["Your", "Passive", "Deployment", "Reaction:", "Once", "Start", "Any", "Effect:", "Declare", "<i>", "Enemy", "Pick"]
        list_results = []
        
        
        
        
        
        
        list_results = []
        for r in results:
            for p in r.split("$"):
                p = p.strip()
                if p and p not in list_results:
                    list_results.append(p)
                    
        
               
        #if x_unique_warscrolls == "/aos4/factions/daughters-of-khaine/Scourge-of-Ghyran-Bloodwrack-Shrine":
        #    print("Skipping Scourge of Ghyran Krethusa the Croneseer",
        #          list_results)
        

        #list_results = list_results[3:-5] # first 4 and last 5 are not abilities and neither are the last 5
        #for x in list_results:   # first 4 and last 5 are not abilities and neither are the last 5
        #    print(x)
        
        
        
        if list_results:
            #print("list_results", list_results)
            for r in list_results:
                if any(r.startswith(start) for start in Acceptable_Starts) and ":" in r:
                    abilities.append(r)
                    
        
            
            # Remove the last element if it contains "Keyword" (usually a footer)
            # This is a workaround for the footer that sometimes appears in the results
            if any("KEYWORDS" in sublist for sublist in abilities):
                abilities.pop()  # removes the last element
                
        #if x_unique_warscrolls == "/aos4/factions/daughters-of-khaine/Krethusa-the-Croneseer":
        #    print("Krethusa-the-Croneseer",  abilities)        
        
        #if x_unique_warscrolls == "/aos4/factions/daughters-of-khaine/Scourge-of-Ghyran-Krethusa-the-Croneseer":
        #    print("Scourge",  abilities)           
                
        
        effect_indices = [i for i, line in enumerate(abilities) if "Effect:" in line]
        effect_indices = [x + 1 for x in effect_indices]
        #for x in abilities:
        #    print(x)
        #print(len(abilities))
        #print("effect_indices", effect_indices)

        
        combined = combine_before_indices(abilities, effect_indices)
        
        for x in combined:
            
            ability_line = re.split(r'(?<!Reaction):| {3,}', x, maxsplit=7)
            ability_line = [item for item in ability_line if item]  
            #for y in ability_line:
                #print(y) 
            
            if is_integer(ability_line[1]):
                joined = ' '.join([ability_line[2], ability_line[1]])
                joined_description = ' '.join(ability_line[4:])
                Unit_Abilities_df.loc[len(Unit_Abilities_df)] = [Warscroll_Name, joined, ability_line[0], ability_line[3], joined_description]
            
            
            #Seperating Choice Abilities 
            elif "<i>" in ability_line[1] or "</i>" in ability_line[1]:
                
                
                
                
                Num_Choice_Abilitys=   sum(item.count("<i>") for item in ability_line)
                
                Big_Line_For_Splitting = ' '.join(ability_line[0:])
                Splitting_Big_Line = [p for p in re.split(r'<i>|</i>', Big_Line_For_Splitting) if p]
                print("Splitting_Big_Line", Splitting_Big_Line)
                print(len(Splitting_Big_Line), "Length of Splitting_Big_Line")
                
                
                counter = 0
                for x in range(0, len(Splitting_Big_Line) - 1, 2):
                    # Combine Name and Splitting_Big_Line[x]
                    Naming = ' '.join([str(Unit_Abilities_df["Name"].iloc[-1 + counter]), str(Splitting_Big_Line[x])])
                    
                    # Combine Description and Splitting_Big_Line[x+1]
                    Description = ' '.join([str(Unit_Abilities_df["Description"].iloc[-1 + counter]), str(Splitting_Big_Line[x + 1])])
                    #Description = str(Splitting_Big_Line[x + 1])
                    
                    # Append new row
                    Unit_Abilities_df.loc[len(Unit_Abilities_df)] = [
                        Unit_Abilities_df["Warscroll"].iloc[-1],  # Warscroll
                        Naming,                                   # Name
                        Unit_Abilities_df["Timing"].iloc[-1],     # Timing
                        Unit_Abilities_df["Flavour_Text"].iloc[-1],  # Flavour_Text
                        Description                                # Description 
                    ]
                    
                    counter -= 1  # increment to move to next row
                
                #Checking if ability at the end of the choices, like what happened with Scourge
                if any(start in Splitting_Big_Line[-1] for start in Acceptable_Starts):
                    # do something
                    text = Splitting_Big_Line[-1]

                    # Build regex pattern for Acceptable_Starts
                    pattern = r'(' + '|'.join(re.escape(word) for word in Acceptable_Starts) + r')'

                    # Search for the first match
                    match = re.search(pattern, text)
                    i = 0
                    if match:
                        start = match.start()
                        print(start, "Start Index")
                        # Keep everything from the first Acceptable_Start to the end
                        Final_Line = text[start:]
                        
                        
                        
                        
                        # Remove the text from the previous entry
                        last_idx = len(Unit_Abilities_df) - 1 
                        # Get current description
                        current_desc = str(Unit_Abilities_df.at[last_idx, "Description"])
                        # Remove Final_Line from the end if it exists there
                        if current_desc.endswith(Final_Line):
                            new_desc = current_desc[: -len(Final_Line)].rstrip()
                        else:
                            new_desc = current_desc  # leave unchanged if not at the end
                        # Update dataframe
                        Unit_Abilities_df.at[last_idx, "Description"] = new_desc
                        
                        
                        
                        
                        Final_Line = re.split(r':| {5,}', Final_Line)
                        Final_Line = [p.strip() for p in Final_Line if p]  # optional: remove empty strings and strip spaces
                        while i < len(Final_Line) - 1:  # stop at the second-to-last element
                            if Final_Line[i].isdigit():  # check if the element is a number
                                Final_Line[i + 1] = Final_Line[i + 1] + " " + Final_Line[i]  # prepend to next
                                Final_Line.pop(i)  # remove the number element
                            else:
                                i += 1  # move to next element only if no pop happened 
                        
                        
                        
                        
                        
                        
                        
                        
                        
                        
                        # Add in the final line
                        
                        joined_description = ' '.join(Final_Line[3:])
                        
                        Unit_Abilities_df.loc[len(Unit_Abilities_df)] = [Warscroll_Name, Final_Line[1], Final_Line[0], Final_Line[2], joined_description]
                
                    else:
                        # No match found, leave as is
                        print("No Acceptable_Start found, last element unchanged.")

                
            else:
                joined_description = ' '.join(ability_line[3:])
                Unit_Abilities_df.loc[len(Unit_Abilities_df)] = [Warscroll_Name, ability_line[1], ability_line[0], ability_line[2], joined_description]
        
        
            
    
    
    
    
    
    
    
    
    #Removing unwanted description of multiuple choice abilities, removes the first instance it appears
    # mark duplicates, but only the *first* occurrence
    # Find duplicated groups
    dup_groups = Unit_Abilities_df.duplicated(
        subset=["Warscroll", "Timing", "Flavour_Text"], keep=False
    )

    # Find the very first occurrence only
    first_occurrence = Unit_Abilities_df.duplicated(
        subset=["Warscroll", "Timing", "Flavour_Text"], keep="first"
    )

    # Keep everything except the very first duplicate
    Unit_Abilities_df = Unit_Abilities_df[~(dup_groups & ~first_occurrence)]
        

    
    
    
    if Text_Cleaning == True:
        # Removing unwanted words from the description, so that it fits on the token
        
        Unit_Abilities_df["Description"] = Unit_Abilities_df["Description"].apply(
        lambda text: removing_unwanted_words(text, parsed_faction_names))
        
        
    if  Passive_Cleaning==True:
        # Removing unwanted words from the description, so that it fits on the token
        
        Unit_Abilities_df = Unit_Abilities_df[~Unit_Abilities_df['Timing'].str.contains('Passive', case=False, na=False)]
    
        
    Unit_Abilities_df.to_csv(f"E:\Licened\Card Tokens\CSVs\{'_'.join(parsed_faction_names)}.csv", index=False)
    return Unit_Abilities_df
    


        
        
        
        
# Example call

Unit_Abilities_df = get_abilities_from_each_warscroll(faction_links[1], faction_names[1], parsed_faction_names[1], Text_Cleaning=True, Passive_Cleaning=True)

Splitting_Big_Line [' Prophecy of Silence  ', ' Until the start of your next turn, enemy units cannot use  commands  while they are in combat with the target. ', ' Prophecy of Dark Wings  ', ' The target can use the ‘  Normal Move  ’ ability as if it were your movement phase. That unit counts as having used a Run ability this turn. ', ' Prophecy of Reclamation  ', ' For the rest of the turn, while the target is contesting an objective  , subtract 10 from the   control   scores   of enemy units contesting that objective that do not have the  HERO  or  MONSTER  keyword.']
6 Length of Splitting_Big_Line
Splitting_Big_Line ['Prophecy of Tyranny ', '  Enemy units cannot use  commands  while they are in combat with the target. ', 'Prophecy of Shelter ', '  Other than the Companion ability,  weapon abilities  used by enemy units while they are in combat with the target have no effect. ', 'Prophecy of Retribution:', '  Subtract 1 from   ward   rolls   made for damage points inflicted by the ta

In [12]:
Unit_Abilities_df

,Warscroll,Name,Timing,Flavour_Text,Description
0,Bloodwrack-Medusa,BLOODWRACK STARE,Passive,Should a victim’s eyes lock with a Bloodwrack ...,Each time this unit attacks with its Bloodw...
1,Bloodwrack-Medusa,MELUSAI KIN,Reaction: You declared a FIGHT ability for t...,A Bloodwrack Medusa leads her Melusai kin into...,Pick a friendly non- HERO MELUSAI unit that...
2,Hag-Queen,WITCHBREW,"Once Per Turn (Army), Any Hero Phase",Witchbrew drives the imbiber into such an ecst...,Pick a friendly DAUGHTERS OF KHAINE unit w...
3,High-Gladiatrix,PARAGON OF SLAUGHTER,Any Combat Phase,The spectacular acts of death-dealing performe...,"If this unit is in combat, pick a visible fr..."
4,High-Gladiatrix,KILLING STROKE,"Once Per Battle, Any Combat Phase",A High Gladiatrix prides herself on slaying en...,Pick an enemy HERO in combat with this uni...
5,Krethusa-the-Croneseer,MURDER OF CROWS 4,Your Hero Phase,Several of Krethusa’s feathers transform into ...,"Pick a visible enemy unit within 18"" of this..."
6,Krethusa-the-Croneseer,BURNT OFFERINGS,"Once Per Turn, Any Hero Phase","Casting blood into her brazier, Krethusa manip...",If this unit is within the combat range of a...
7,Krethusa-the-Croneseer,BURNT OFFERINGS Prophecy of Silence,"Once Per Turn, Any Hero Phase","Casting blood into her brazier, Krethusa manip...","Casting blood into her brazier, Krethusa manip..."
8,Krethusa-the-Croneseer,BURNT OFFERINGS Prophecy of Dark Wings,"Once Per Turn, Any Hero Phase","Casting blood into her brazier, Krethusa manip...","Casting blood into her brazier, Krethusa manip..."
9,Krethusa-the-Croneseer,BURNT OFFERINGS Prophecy of Reclamation,"Once Per Turn, Any Hero Phase","Casting blood into her brazier, Krethusa manip...","Casting blood into her brazier, Krethusa manip..."


In [ ]:


Unit_Abilities_df["Description"] = Unit_Abilities_df["Description"].apply(
    lambda text: removing_unwanted_words(text, parsed_faction_names)
)
print(Unit_Abilities_df["Description"])

0     Each time attacks with its Bloodwrack Stare at...
1     Pick non HERO MELUSAI not used FIGHT ability s...
2                     Pick wholly 12 dice On 3+ WARD 5+
3     combat pick visible non HERO DAUGHTERS KHAINE ...
4     Pick an enemy HERO combat with 2D6 exceeds s H...
5     Pick visible enemy 18 then make chanting D6 D3...
6     combat CAULDRON BLOOD pick visible non HERO DA...
7     Casting blood into her brazier Krethusa manipu...
8     Casting blood into her brazier Krethusa manipu...
9     Casting blood into her brazier Krethusa manipu...
10    Pick non HERO MELUSAI not used FIGHT ability s...
11                can re charge MELUSAI units wholly 12
12    Shadow Queen on battlefield make casting 2D6 P...
13    Each time damage point would allocated it inst...
14                                           +1 casting
15    Pick AELF INFANTRY wholly 12 dice On 3+ add 1 ...
16                                Give D3 ritual points
17    Pick visible DAUGHTERS KHAINE AELF INFANTR

In [16]:
print(Unit_Abilities_df["Description"][0])

Each time attacks with its Bloodwrack Stare attack scores hit number dice equal number models each 5+ inflict 1 mortal damage on cannot pick same enemy targeted by attacks made with Bloodwrack Stare more than once


In [ ]:


    
    
    
    
    
    
    
    
    if len(parts) < 2:
                continue  # skip malformed lines

            # Timing
            #for x in range(len(parts)):
            #    parts[x] = removing_unwanted_words(parts[x], parsed_faction_names)
                
            Timing = parts[0]
            
            
            # Name
            Name = parts[1]
            
            


            
            if is_integer(Name):
                # Description
                #print(parts)
                
                Description = removing_unwanted_words(' '.join(parts[3:]) if len(parts) > 2 else "" , parsed_faction_names)
                
                Keywords = parts[-1]
                
                for idx, part in enumerate(parts):
                    sub_soup = BeautifulSoup(part, "html.parser")
                    italics = [tag.get_text() for tag in sub_soup.find_all(["i", "em"])]
                
                    if italics:
                        print(f"Part {idx} has italics: {italics}")
                    else:
                        print(f"Part {idx} has no italics")
                
                Description = ' '.join([Name, Description])
                
                Unit_Abilities_df.loc[len(Unit_Abilities_df)] = [Warscroll_Name, Timing, parts[2], Description, Keywords]
            elif "Reaction" in Timing:
                Timing = ' '.join([parts[0], parts[1]])
                Description = removing_unwanted_words(' '.join(parts[4:]) if len(parts) > 2 else "" , parsed_faction_names)
                Unit_Abilities_df.loc[len(Unit_Abilities_df)] = [Warscroll_Name, Timing, parts[3], Description, ""]
                
            
            
            else:
                
                # Add to DataFrame
                # Description
                Description = removing_unwanted_words(' '.join(parts[2:]) if len(parts) > 2 else "" , parsed_faction_names)
                Unit_Abilities_df.loc[len(Unit_Abilities_df)] = [Warscroll_Name, Timing, Name, Description, ""]






        # Save DataFrame to CSV
        Unit_Abilities_df.to_csv(f"E:\Licened\Card Tokens\CSVs\{'_'.join(parsed_faction_names)}.csv", index=False)
        
        
        #Unit_Abilities_df_Filtered = Unit_Abilities_df[~Unit_Abilities_df["Timing"].str.contains("Passive", case=False, na=False)]
        #Unit_Abilities_df_Filtered.to_csv(f"E:\Licened\Card Tokens\CSVs\{'_'.join(parsed_faction_names)}_Filtered_Passive.csv", index=False)
    
        # Save DataFrame to CSV
        
    return Unit_Abilities_df